# VerityOps — Hallucination / Risk Classifier: EDA & Modeling Walkthrough

**All data is synthetic** (generated by `src/generate_data.py`, seed 42). Run from the project root with the project virtual environment active. This notebook mirrors `src/train.py` in interactive form; the authoritative run is `python run_all.py`.

In [ ]:
import pandas as pd, numpy as np
from src.features import FEATURE_COLUMNS, build_feature_frame
from src.generate_data import generate
raw = generate(n=2400, seed=42)
raw.head(3)[['work_order_id','device_model','llm_summary','is_hallucinated','corruption_types']]

## 1. Class balance and the confidence trap
~30% of summaries are hallucinated. Note the LLM's self-reported confidence overlaps heavily between classes — overconfident hallucinations exist, so confidence alone must never auto-approve.

In [ ]:
print('n =', len(raw), '| prevalence =', round(raw['is_hallucinated'].mean(), 3))
print(raw.groupby('is_hallucinated')['llm_confidence'].describe()[['mean','std','min','max']].round(3))
print('\nCorruption mix (hallucinated rows):')
print(raw.loc[raw['is_hallucinated']==1, 'corruption_types'].str.split(';').explode().value_counts())

## 2. Features — grounding checks, not vibes
Each feature compares the summary against the structured source work order. Crucially, every check is split into *stated* vs *match*: a field the summary never mentions is missing evidence, not evidence of error.

In [ ]:
feats = build_feature_frame(raw)
print('features:', len(FEATURE_COLUMNS))
feats[FEATURE_COLUMNS].describe().T[['mean','std','min','max']].round(3)

In [ ]:
# Do grounding checks separate the classes? (means by label)
feats.groupby('is_hallucinated')[['num_mismatch_count','cost_delta_pct','hours_delta',
    'part_code_match','date_match','fields_stated_count','hedge_count']].mean().round(3)

## 3. Train / compare — rules baseline vs Logistic Regression vs Random Forest vs Gradient Boosting vs text-only
Thresholds are tuned for max F1 on the validation split (n=360), then scored once on test (n=360).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
X = feats[FEATURE_COLUMNS].to_numpy(); y = feats['is_hallucinated'].to_numpy()
idx = np.arange(len(feats))
tr, tmp = train_test_split(idx, test_size=720, random_state=42, stratify=y)
va, te = train_test_split(tmp, test_size=360, random_state=42, stratify=y[tmp])
def f1_at(yv, p, t=0.5): return f1_score(yv, (p >= t).astype(int))
models = {
 'Logistic Regression': Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=2000, random_state=42))]),
 'Random Forest': RandomForestClassifier(n_estimators=300, min_samples_leaf=2, n_jobs=-1, random_state=42, class_weight='balanced'),
 'Gradient Boosting': HistGradientBoostingClassifier(max_iter=300, learning_rate=0.06, random_state=42),
}
for m in models.values(): m.fit(X[tr], y[tr])
for name, m in models.items():
    p_va = m.predict_proba(X[va])[:, 1]; t = max(np.arange(0.05, 0.96, 0.05), key=lambda th: f1_at(y[va], p_va, th))
    p_te = m.predict_proba(X[te])[:, 1]
    print(f"{name:22s} thr={t:.2f} AUC={roc_auc_score(y[te], p_te):.4f} P={precision_score(y[te], p_te>=t):.4f} R={recall_score(y[te], p_te>=t):.4f} F1={f1_score(y[te], p_te>=t):.4f}")

## 4. Where the model is weak — recall by corruption type
The nine hard-contradiction types are caught at recall 1.000. The misses are the subtle fabrications (wrong part name, plausible-but-wrong problem description) and invented costs hidden by field omission. That is the honest headline: near-perfect contradiction detector, weak subtle-fabrication detector — human audit stays in the loop.

In [ ]:
import joblib
b = joblib.load('models/best_model.pkl')
te_df = feats.iloc[te].copy()
te_df['pred'] = (b['model'].predict_proba(X[te])[:, 1] >= b['threshold']).astype(int)
hall = te_df[te_df['is_hallucinated'] == 1].copy()
hall['ctype'] = hall['corruption_types'].str.split(';')
print(hall.explode('ctype').groupby('ctype').apply(
    lambda g: pd.Series({'n': len(g), 'recall': round(float((g['pred'] == 1).mean()), 3)}),
    include_groups=False).sort_values('recall').to_string())

## 5. Monitoring — the detection gap
`src/monitor.py` scores a fresh 90-day stream (n=2,700) and tracks hallucination rate, precision/recall with a 7-day audit lag, and the detection gap (audited hallucination rate − flag rate). During a simulated generator upgrade (days 45–59) recall falls 0.648 → 0.534 while precision stays 0.86 — the gap, not precision, catches it. See `reports/monitoring.md` and `reports/figures/monitoring.png`.

In [ ]:
import json
mon = json.load(open('models/monitoring_metrics.json'))
print('90-day stream: hall rate', mon['overall_hallucination_rate'], '| flag rate', mon['overall_flag_rate'],
      '| precision', mon['overall_precision'], '| recall', mon['overall_recall'])
print('alert days:', mon['alert_days'], '| drift recall', mon['drift_episode_days_45_59_recall'],
      'vs baseline', mon['baseline_days_0_44_recall'], '| drift gap', mon['drift_episode_days_45_59_gap'],
      'vs baseline', mon['baseline_days_0_44_gap'])